# NIPS CITE-seq processed objects -> R-ready CSV

This notebook converts the outputs from `nips_preprocess_clean.ipynb` into plain CSV files for downstream R scripts.

Outputs are written to `processed/csv_for_R/`:

| File | Contents |
|---|---|
| `cite_nips_RNA_for_R.csv` | one row per cell, metadata columns first, then processed RNA HVG features |
| `cite_nips_ADT_for_R.csv` | one row per cell, metadata columns first, then processed ADT protein features |
| `cite_nips_obs_metadata_for_R.csv` | observation metadata only |
| `cite_nips_feature_name_map_for_R.csv` | original feature names and their R-safe CSV column names |
| `cite_nips_hvg_genes_for_R.csv` | HVG list with matching R-safe RNA column names |

The exported matrices use observations/cells as rows. The first columns are `obs_id`, `batch`, `cell_type`, and `cell_type_major`; feature columns are made syntactically safe and unique for R.

In [1]:
# If needed in a fresh environment:
# !pip install anndata pandas scipy

from pathlib import Path
import re

import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse

## 1. Configure paths and export settings

`RNA_LAYER = None` and `ADT_LAYER = None` export the current `.X` matrices from the processed objects. In the preprocessing notebook, this means scaled/log-normalized HVG RNA in `.X` and CLR-normalized ADT in `.X`. You can change `RNA_LAYER` to `'lognorm'` or `ADT_LAYER` to `'clr'` if a different stored layer is preferred.

In [2]:
BASE_DIR_CANDIDATES = [Path("."), Path("singlecell")]
BASE_DIR = next(
    (
        path
        for path in BASE_DIR_CANDIDATES
        if (path / "processed" / "cite_nips_RNA_proc.h5ad").exists()
        and (path / "processed" / "cite_nips_ADT_proc.h5ad").exists()
    ),
    None,
)
assert BASE_DIR is not None, "Could not find processed h5ad files from '.' or './singlecell'."
PROCESSED_DIR = BASE_DIR / "processed"
OUT_DIR = PROCESSED_DIR / "csv_for_R"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RNA_H5AD = PROCESSED_DIR / "cite_nips_RNA_proc.h5ad"
ADT_H5AD = PROCESSED_DIR / "cite_nips_ADT_proc.h5ad"
HVG_CSV = PROCESSED_DIR / "cite_nips_hvg_genes.csv"

RNA_LAYER = None       # None exports adata_rna.X; use "lognorm" for unscaled log-normalized RNA if desired
ADT_LAYER = None       # None exports adata_adt.X; use "clr" if desired
CHUNK_SIZE = 5000      # number of cells written per chunk
FLOAT_FORMAT = "%.7g" # compact numeric text, usually enough for downstream modeling

print("RNA input:", RNA_H5AD.resolve())
print("ADT input:", ADT_H5AD.resolve())
print("CSV output directory:", OUT_DIR.resolve())

RNA input: /Users/keyao/Code/GitHub/DORM/singlecell/processed/cite_nips_RNA_proc.h5ad
ADT input: /Users/keyao/Code/GitHub/DORM/singlecell/processed/cite_nips_ADT_proc.h5ad
CSV output directory: /Users/keyao/Code/GitHub/DORM/singlecell/processed/csv_for_R


## 2. Load processed AnnData objects and validate pairing

In [3]:
adata_rna = ad.read_h5ad(RNA_H5AD)
adata_adt = ad.read_h5ad(ADT_H5AD)

print(adata_rna)
print(adata_adt)

assert adata_rna.n_obs == adata_adt.n_obs, "RNA and ADT objects have different numbers of cells."
assert adata_rna.obs_names.equals(adata_adt.obs_names), "RNA and ADT obs_names are not aligned."
assert "batch" in adata_rna.obs.columns, "RNA obs must contain a 'batch' column."

print("Paired cells:", adata_rna.n_obs)
print("RNA features:", adata_rna.n_vars)
print("ADT features:", adata_adt.n_vars)
print("RNA obs columns:", list(adata_rna.obs.columns))

AnnData object with n_obs × n_vars = 90261 × 1000
    obs: 'GEX_n_genes_by_counts', 'GEX_pct_counts_mt', 'GEX_size_factors', 'GEX_phase', 'ADT_n_antibodies_by_counts', 'ADT_total_counts', 'ADT_iso_count', 'cell_type', 'batch', 'ADT_pseudotime_order', 'GEX_pseudotime_order', 'Samplename', 'Site', 'DonorNumber', 'Modality', 'VendorLot', 'DonorID', 'DonorAge', 'DonorBMI', 'DonorBloodType', 'DonorRace', 'Ethnicity', 'DonorGender', 'QCMeds', 'DonorSmoker', 'is_train', 'cell_type_major'
    var: 'feature_types', 'gene_id'
    uns: 'dataset_id', 'genome', 'log1p', 'organism', 'preprocessing'
    obsm: 'ADT_X_pca', 'ADT_X_umap', 'ADT_isotype_controls', 'GEX_X_pca', 'GEX_X_umap'
    layers: 'counts', 'lognorm'
AnnData object with n_obs × n_vars = 90261 × 134
    obs: 'GEX_n_genes_by_counts', 'GEX_pct_counts_mt', 'GEX_size_factors', 'GEX_phase', 'ADT_n_antibodies_by_counts', 'ADT_total_counts', 'ADT_iso_count', 'cell_type', 'batch', 'ADT_pseudotime_order', 'GEX_pseudotime_order', 'Samplename', '

## 3. Ensure major cell type is present

The preprocessing notebook creates `cell_type_major`. This cell recomputes it from `cell_type` only if it is missing, so the export can still run from the saved processed objects.

In [4]:
cell_library = {
    "monocyte": ["CD14+ Mono", "CD16+ Mono"],
    "T cell": [
        "CD4+ T naive", "CD4+ T activated", "CD4+ T activated integrinB7+",
        "T reg", "CD8+ T naive", "CD8+ T naive CD127+ CD26- CD101-",
        "CD8+ T CD49f+", "CD8+ T TIGIT+ CD45RO+", "CD8+ T CD57+ CD45RA+",
        "CD8+ T CD69+ CD45RO+", "CD8+ T TIGIT+ CD45RA+", "CD8+ T CD69+ CD45RA+",
        "CD8+ T CD57+ CD45RO+", "CD4+ T CD314+ CD45RA+", "dnT", "MAIT",
        "gdT CD158b+", "gdT TCRVD2+",
    ],
    "NK/ILC": ["NK", "NK CD158e1+", "ILC", "ILC1"],
    "B cell": [
        "Naive CD20+ B IGKC+", "Naive CD20+ B IGKC-", "B1 B IGKC-",
        "B1 B IGKC+", "Transitional B", "Plasmablast IGKC+",
        "Plasmablast IGKC-", "Plasma cell IGKC+", "Plasma cell IGKC-",
    ],
    "DC": ["cDC1", "cDC2", "pDC"],
    "Progenitor": ["HSC", "Lymph prog", "G/M prog", "MK/E prog", "T prog cycling"],
    "Erythroid": ["Erythroblast", "Proerythroblast", "Normoblast", "Reticulocyte"],
}

fine2major = {fine: major for major, labels in cell_library.items() for fine in labels}

def ensure_major_cell_type(adata_obj):
    if "cell_type_major" not in adata_obj.obs.columns:
        assert "cell_type" in adata_obj.obs.columns, "Need 'cell_type' to rebuild 'cell_type_major'."
        adata_obj.obs["cell_type_major"] = adata_obj.obs["cell_type"].map(fine2major).fillna("Other")
    return adata_obj

adata_rna = ensure_major_cell_type(adata_rna)
adata_adt = ensure_major_cell_type(adata_adt)

print(adata_rna.obs["cell_type_major"].value_counts(dropna=False))

cell_type_major
T cell        26961
monocyte      24328
Erythroid     11258
B cell         9866
NK/ILC         8391
Progenitor     5979
DC             3478
Name: count, dtype: int64


## 4. Build R-safe feature names and metadata

R's default `read.csv()` applies `make.names()` unless `check.names = FALSE`. To keep downstream behavior predictable, this notebook exports feature columns that are already safe, unique, and stable. The mapping file preserves the original gene/protein names.

In [5]:
META_COLUMNS = ["obs_id", "batch", "cell_type", "cell_type_major"]

def make_r_safe_names(names, prefix):
    """Create unique column names that are friendly to R formulas and read.csv()."""
    safe = []
    used = set(META_COLUMNS)
    for i, name in enumerate(map(str, names), start=1):
        candidate = re.sub(r"[^0-9A-Za-z_.]", "_", name).strip("_")
        candidate = re.sub(r"_+", "_", candidate)
        if not candidate:
            candidate = f"feature_{i}"
        if not re.match(r"^[A-Za-z]", candidate):
            candidate = f"{prefix}_{candidate}"
        base = candidate
        j = 2
        while candidate in used:
            candidate = f"{base}_{j}"
            j += 1
        used.add(candidate)
        safe.append(candidate)
    return safe

rna_feature_cols = make_r_safe_names(adata_rna.var_names, "rna")
adt_feature_cols = make_r_safe_names(adata_adt.var_names, "adt")

obs_meta = pd.DataFrame({
    "obs_id": adata_rna.obs_names.astype(str),
    "batch": adata_rna.obs["batch"].astype(str).to_numpy(),
    "cell_type": adata_rna.obs["cell_type"].astype(str).to_numpy() if "cell_type" in adata_rna.obs.columns else "NA",
    "cell_type_major": adata_rna.obs["cell_type_major"].astype(str).to_numpy(),
})

feature_map = pd.concat(
    [
        pd.DataFrame({
            "modality": "RNA",
            "feature_index": np.arange(1, adata_rna.n_vars + 1),
            "original_name": adata_rna.var_names.astype(str),
            "csv_name": rna_feature_cols,
        }),
        pd.DataFrame({
            "modality": "ADT",
            "feature_index": np.arange(1, adata_adt.n_vars + 1),
            "original_name": adata_adt.var_names.astype(str),
            "csv_name": adt_feature_cols,
        }),
    ],
    ignore_index=True,
)

print(obs_meta.head())
print(feature_map.head())
print(feature_map.tail())

                    obs_id batch            cell_type cell_type_major
0  GCATTAGCATAAGCGG-1-s1d1  s1d1  Naive CD20+ B IGKC+          B cell
1  TACAGGTGTTAGAGTA-1-s1d1  s1d1           CD14+ Mono        monocyte
2  AGGATCTAGGTCTACT-1-s1d1  s1d1  Naive CD20+ B IGKC+          B cell
3  GTAGAAAGTGACACAG-1-s1d1  s1d1                  HSC      Progenitor
4  TCCGAAAAGGATCATA-1-s1d1  s1d1         Reticulocyte       Erythroid
  modality  feature_index original_name csv_name
0      RNA              1          HES4     HES4
1      RNA              2         ISG15    ISG15
2      RNA              3         SMIM1    SMIM1
3      RNA              4          RBP7     RBP7
4      RNA              5           SRM      SRM
     modality  feature_index original_name csv_name
1129      ADT            130       HLA-E-1  HLA_E_1
1130      ADT            131        CD82-1   CD82_1
1131      ADT            132       CD101-1  CD101_1
1132      ADT            133          CD88     CD88
1133      ADT            1

## 5. Export helper

The helper writes the matrix in row chunks, which avoids building one very large dense data frame in memory.

In [6]:
def get_matrix(adata_obj, layer):
    if layer is None:
        return adata_obj.X
    assert layer in adata_obj.layers, f"Layer {layer!r} not found. Available layers: {list(adata_obj.layers.keys())}"
    return adata_obj.layers[layer]

def dense_block(matrix, start, stop):
    block = matrix[start:stop, :]
    if sparse.issparse(block):
        block = block.toarray()
    return np.asarray(block)

def export_matrix_csv(adata_obj, layer, feature_cols, metadata, out_path, chunksize=5000):
    matrix = get_matrix(adata_obj, layer)
    n_obs = adata_obj.n_obs
    wrote_header = False

    for start in range(0, n_obs, chunksize):
        stop = min(start + chunksize, n_obs)
        expr = dense_block(matrix, start, stop)
        expr_df = pd.DataFrame(expr, columns=feature_cols)
        chunk = pd.concat(
            [metadata.iloc[start:stop].reset_index(drop=True), expr_df],
            axis=1,
        )
        chunk.to_csv(
            out_path,
            mode="w" if not wrote_header else "a",
            index=False,
            header=not wrote_header,
            float_format=FLOAT_FORMAT,
        )
        wrote_header = True
        print(f"{out_path.name}: wrote rows {start + 1}-{stop} of {n_obs}")

    return out_path

## 6. Write CSV files

In [8]:
obs_meta_path = OUT_DIR / "cite_nips_obs_metadata_for_R.csv"
feature_map_path = OUT_DIR / "cite_nips_feature_name_map_for_R.csv"
rna_csv_path = OUT_DIR / "cite_nips_RNA_for_R.csv"
adt_csv_path = OUT_DIR / "cite_nips_ADT_for_R.csv"
hvg_out_path = OUT_DIR / "cite_nips_hvg_genes_for_R.csv"

obs_meta.to_csv(obs_meta_path, index=False)
feature_map.to_csv(feature_map_path, index=False)

if HVG_CSV.exists():
    hvg = pd.read_csv(HVG_CSV)
    hvg_name_col = "hvg_gene" if "hvg_gene" in hvg.columns else hvg.columns[0]
    hvg = hvg.rename(columns={hvg_name_col: "hvg_gene"})
    hvg = hvg.merge(
        feature_map.loc[feature_map["modality"] == "RNA", ["original_name", "csv_name"]],
        left_on="hvg_gene",
        right_on="original_name",
        how="left",
    ).drop(columns=["original_name"])
    hvg.to_csv(hvg_out_path, index=False)
else:
    pd.DataFrame({"hvg_gene": adata_rna.var_names.astype(str), "csv_name": rna_feature_cols}).to_csv(hvg_out_path, index=False)

export_matrix_csv(adata_rna, RNA_LAYER, rna_feature_cols, obs_meta, rna_csv_path, chunksize=CHUNK_SIZE)
export_matrix_csv(adata_adt, ADT_LAYER, adt_feature_cols, obs_meta, adt_csv_path, chunksize=CHUNK_SIZE)

print("Done.")
print("RNA CSV:", rna_csv_path.resolve())
print("ADT CSV:", adt_csv_path.resolve())

cite_nips_RNA_for_R.csv: wrote rows 1-5000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 5001-10000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 10001-15000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 15001-20000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 20001-25000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 25001-30000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 30001-35000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 35001-40000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 40001-45000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 45001-50000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 50001-55000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 55001-60000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 60001-65000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 65001-70000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 70001-75000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 75001-80000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 80001-85000 of 90261
cite_nips_RNA_for_R.csv: wrote rows 8

## 7. Quick validation and R usage notes

In [9]:
rna_preview = pd.read_csv(rna_csv_path, nrows=3)
adt_preview = pd.read_csv(adt_csv_path, nrows=3)

assert list(rna_preview.columns[:4]) == META_COLUMNS
assert list(adt_preview.columns[:4]) == META_COLUMNS
assert rna_preview.shape[1] == len(META_COLUMNS) + adata_rna.n_vars
assert adt_preview.shape[1] == len(META_COLUMNS) + adata_adt.n_vars

print("RNA preview shape:", rna_preview.shape)
print("ADT preview shape:", adt_preview.shape)
display(rna_preview.iloc[:, :8])
display(adt_preview.iloc[:, :8])

print("In R:")
print('rna <- read.csv("processed/csv_for_R/cite_nips_RNA_for_R.csv", check.names = FALSE)')
print('adt <- read.csv("processed/csv_for_R/cite_nips_ADT_for_R.csv", check.names = FALSE)')
print('rna_x <- as.matrix(rna[, !(names(rna) %in% c("obs_id", "batch", "cell_type", "cell_type_major"))])')
print('adt_x <- as.matrix(adt[, !(names(adt) %in% c("obs_id", "batch", "cell_type", "cell_type_major"))])')

RNA preview shape: (3, 1004)
ADT preview shape: (3, 138)


,obs_id,batch,cell_type,cell_type_major,HES4,ISG15,SMIM1,RBP7
0,GCATTAGCATAAGCGG-1-s1d1,s1d1,Naive CD20+ B IGKC+,B cell,-0.143117,-0.59474,5.606847,-0.354408
1,TACAGGTGTTAGAGTA-1-s1d1,s1d1,CD14+ Mono,monocyte,-0.143117,-0.59474,-0.275130,-0.354408
2,AGGATCTAGGTCTACT-1-s1d1,s1d1,Naive CD20+ B IGKC+,B cell,-0.143117,-0.59474,-0.275130,-0.354408


,obs_id,batch,cell_type,cell_type_major,CD86_1,CD274,CD270,CD155
0,GCATTAGCATAAGCGG-1-s1d1,s1d1,Naive CD20+ B IGKC+,B cell,0.388389,0.172376,0.891182,0.00000
1,TACAGGTGTTAGAGTA-1-s1d1,s1d1,CD14+ Mono,monocyte,3.299652,0.840217,1.485070,2.42784
2,AGGATCTAGGTCTACT-1-s1d1,s1d1,Naive CD20+ B IGKC+,B cell,0.000000,0.319362,1.120746,0.00000


In R:
rna <- read.csv("processed/csv_for_R/cite_nips_RNA_for_R.csv", check.names = FALSE)
adt <- read.csv("processed/csv_for_R/cite_nips_ADT_for_R.csv", check.names = FALSE)
rna_x <- as.matrix(rna[, !(names(rna) %in% c("obs_id", "batch", "cell_type", "cell_type_major"))])
adt_x <- as.matrix(adt[, !(names(adt) %in% c("obs_id", "batch", "cell_type", "cell_type_major"))])
